### Load our configuration

In [1]:
from pathlib import Path
import sys
import yaml
import torch
import torchvision
import timm

# Find project root
ROOT = Path.cwd().resolve()

if ROOT.name == "notebooks":
    ROOT = ROOT.parent

sys.path.insert(0, str(ROOT / "src"))

from vehicle_data import VehicleManifestDataset

with open(ROOT / "configs/experiments.yaml") as f:
    cfg = yaml.safe_load(f)

print("Project:", cfg["project_name"])
print("Seed:", cfg["random_seed"])
print("PyTorch:", torch.__version__)
print("Torchvision:", torchvision.__version__)
print("timm:", timm.__version__)

Project: vehicle-make-model-recognition
Seed: 42
PyTorch: 2.15.0
Torchvision: 0.29.2
timm: 1.0.30


/Users/bata/Documents/vehicle-make-model-recognition/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Step 2 — Initialize Xception with pretrained weights

In [2]:
model_key = "xception"
dataset_key = "pakistan"

model_name = cfg["models"][model_key]["timm_name"]
num_classes = cfg["datasets"][dataset_key]["verified_classes"]

model = timm.create_model(
    model_name,
    pretrained=True,
    num_classes=num_classes
)

model.eval()

print("Model:", model_name)
print("Output classes:", num_classes)
print("Feature dimension:", model.num_features)
print("Classifier:", model.get_classifier())

Downloading: "https://github.com/rwightman/pytorch-image-models/releases/download/v0.1-cadene/xception-43020ad28.pth" to /Users/bata/.cache/torch/hub/checkpoints/xception-43020ad28.pth
Model: legacy_xception.tf_in1k
Output classes: 48
Feature dimension: 2048
Classifier: Linear(in_features=2048, out_features=48, bias=True)


# Step 3 — Count model parameters

In [3]:
total_params = sum(
    p.numel() for p in model.parameters()
)

trainable_params = sum(
    p.numel()
    for p in model.parameters()
    if p.requires_grad
)

print(f"Total parameters: {total_params:,}")
print(f"Trainable parameters: {trainable_params:,}")

Total parameters: 20,905,304
Trainable parameters: 20,905,304


# Step 4 — Prepare one real validation image

In [4]:
from vehicle_transforms import build_vehicle_transforms

train_transform, eval_transform = build_vehicle_transforms(
    cfg, model
)

val_dataset = VehicleManifestDataset(
    ROOT,
    dataset_key,
    "val",
    transform=eval_transform
)

image, label = val_dataset[0]

print("Image shape:", image.shape)
print("Class index:", label)
print("Class name:", val_dataset.classes[label])

Image shape: torch.Size([3, 224, 224])
Class index: 0
Class name: Daiatsu_Core


## 5. Perform our first forward pass

In [5]:
x = image.unsqueeze(0)

print("Input shape:", x.shape)

with torch.inference_mode():
    output = model(x)

print("Output shape:", output.shape)

assert output.shape == (1, num_classes)
assert torch.isfinite(output).all()

print("Xception forward pass: PASS")

Input shape: torch.Size([1, 3, 224, 224])
Output shape: torch.Size([1, 48])
Xception forward pass: PASS


In [6]:
# Freeze all parameters
for p in model.parameters():
    p.requires_grad = False

# Unfreeze only the classifier
for p in model.get_classifier().parameters():
    p.requires_grad = True

total = sum(p.numel() for p in model.parameters())
trainable = sum(
    p.numel()
    for p in model.parameters()
    if p.requires_grad
)

assert 0 < trainable < total

print(f"Total parameters: {total:,}")
print(f"Trainable parameters: {trainable:,}")
print(f"Frozen parameters: {total - trainable:,}")

Total parameters: 20,905,304
Trainable parameters: 98,352
Frozen parameters: 20,806,952


# 2. Next: Initialize ConvNeXt-Tiny on Pakistani vehicles (E2)

#### Step 1 — Release Xception from memory

In [7]:
import gc

del model
gc.collect()

print("Previous model released")

Previous model released


### Step 2 — Initialize ConvNeXt-Tiny

In [8]:
model_key = "convnext_tiny"
dataset_key = "pakistan"

model_name = cfg["models"][model_key]["timm_name"]
num_classes = cfg["datasets"][dataset_key]["verified_classes"]

model = timm.create_model(
    model_name,
    pretrained=True,
    num_classes=num_classes
)

model.eval()

print("Model:", model_name)
print("Output classes:", num_classes)
print("Feature dimension:", model.num_features)
print("Classifier:", model.get_classifier())

total_params = sum(p.numel() for p in model.parameters())

print(f"Total parameters: {total_params:,}")

Model: convnext_tiny.fb_in1k
Output classes: 48
Feature dimension: 768
Classifier: Linear(in_features=768, out_features=48, bias=True)
Total parameters: 27,857,040


#### Step 3 — Apply ConvNeXt's preprocessing

In [9]:
from vehicle_transforms import build_vehicle_transforms
from timm.data import resolve_model_data_config

train_transform, eval_transform = build_vehicle_transforms(
    cfg, model
)

model_data = resolve_model_data_config(model)

print("Mean:", model_data["mean"])
print("Std:", model_data["std"])

val_dataset = VehicleManifestDataset(
    ROOT,
    dataset_key,
    "val",
    transform=eval_transform
)

image, label = val_dataset[0]

print("Image shape:", image.shape)
print("Class index:", label)
print("Class name:", val_dataset.classes[label])

Mean: (0.485, 0.456, 0.406)
Std: (0.229, 0.224, 0.225)
Image shape: torch.Size([3, 224, 224])
Class index: 0
Class name: Daiatsu_Core


### 4. Run Forward Pass

In [10]:
x = image.unsqueeze(0)

with torch.inference_mode():
    output = model(x)

print("Input shape:", x.shape)
print("Output shape:", output.shape)

assert output.shape == (1, 48)
assert torch.isfinite(output).all()

print("ConvNeXt-Tiny forward pass: PASS")

Input shape: torch.Size([1, 3, 224, 224])
Output shape: torch.Size([1, 48])
ConvNeXt-Tiny forward pass: PASS


#### Complete ConvNeXt-Tiny's forward pass

In [11]:
# Prepare one image as a batch
x = image.unsqueeze(0)

# Forward pass without calculating gradients
model.eval()

with torch.inference_mode():
    output = model(x)

print("Input shape:", x.shape)
print("Output shape:", output.shape)

assert output.shape == (1, 48)
assert torch.isfinite(output).all()

print("ConvNeXt-Tiny forward pass: PASS")

Input shape: torch.Size([1, 3, 224, 224])
Output shape: torch.Size([1, 48])
ConvNeXt-Tiny forward pass: PASS


# Stanford Cars

### Step 1 — Initialize Xception for Stanford Cars (E3)

In [12]:
import gc

del model
gc.collect()

40

In [13]:
model_key = "xception"
dataset_key = "stanford_cars"

model_name = cfg["models"][model_key]["timm_name"]
num_classes = cfg["datasets"][dataset_key]["verified_classes"]

torch.manual_seed(cfg["random_seed"])

model = timm.create_model(
    model_name,
    pretrained=True,
    num_classes=num_classes
)

model.eval()

print("Model:", model_name)
print("Classes:", num_classes)
print("Classifier:", model.get_classifier())

total_params = sum(p.numel() for p in model.parameters())
print(f"Total parameters: {total_params:,}")

Model: legacy_xception.tf_in1k
Classes: 196
Classifier: Linear(in_features=2048, out_features=196, bias=True)
Total parameters: 21,208,556


### Step 2 — Verify Stanford Cars preprocessing and forward pass

In [14]:
train_transform, eval_transform = build_vehicle_transforms(
    cfg, model
)

val_dataset = VehicleManifestDataset(
    ROOT,
    dataset_key,
    "val",
    transform=eval_transform
)

assert len(val_dataset) == 1214

image, label = val_dataset[0]

print("Image shape:", image.shape)
print("Class:", val_dataset.classes[label])

x = image.unsqueeze(0)

with torch.inference_mode():
    output = model(x)

print("Output shape:", output.shape)

assert output.shape == (1, 196)
assert torch.isfinite(output).all()

print("Xception Stanford Cars: PASS")

Image shape: torch.Size([3, 224, 224])
Class: AM General Hummer SUV 2000
Output shape: torch.Size([1, 196])
Xception Stanford Cars: PASS


## Step 3 — Repeat for ConvNeXt-Tiny (E4)

In [15]:
del model
gc.collect()

7

In [16]:
model_key = "convnext_tiny"
dataset_key = "stanford_cars"

model_name = cfg["models"][model_key]["timm_name"]
num_classes = cfg["datasets"][dataset_key]["verified_classes"]

torch.manual_seed(cfg["random_seed"])

model = timm.create_model(
    model_name,
    pretrained=True,
    num_classes=num_classes
)

model.eval()

print("Model:", model_name)
print("Classes:", num_classes)
print("Classifier:", model.get_classifier())

total_params = sum(p.numel() for p in model.parameters())
print(f"Total parameters: {total_params:,}")

Model: convnext_tiny.fb_in1k
Classes: 196
Classifier: Linear(in_features=768, out_features=196, bias=True)
Total parameters: 27,970,852


In [17]:
train_transform, eval_transform = build_vehicle_transforms(
    cfg, model
)

val_dataset = VehicleManifestDataset(
    ROOT,
    dataset_key,
    "val",
    transform=eval_transform
)

assert len(val_dataset) == 1214

image, label = val_dataset[0]

print("Image shape:", image.shape)
print("Class:", val_dataset.classes[label])

x = image.unsqueeze(0)

with torch.inference_mode():
    output = model(x)

print("Output shape:", output.shape)

assert output.shape == (1, 196)
assert torch.isfinite(output).all()

print("Xception Stanford Cars: PASS")

Image shape: torch.Size([3, 224, 224])
Class: AM General Hummer SUV 2000
Output shape: torch.Size([1, 196])
Xception Stanford Cars: PASS


## Freezing and Fine-Tuning Setup

## Verify Stage 1 — Head-only training setup

In [18]:
from vehicle_models import configure_transfer_stage

In [19]:
print("Architecture:", model_key)
print("Dataset:", dataset_key)
print("Classifier:", model.get_classifier())

Architecture: convnext_tiny
Dataset: stanford_cars
Classifier: Linear(in_features=768, out_features=196, bias=True)


In [20]:
stats = configure_transfer_stage(
    model,
    model_key,
    stage="head_only"
)

print("Total parameters:", f"{stats['total']:,}")
print("Trainable parameters:", f"{stats['trainable']:,}")
print("Frozen parameters:", f"{stats['frozen']:,}")

assert stats["trainable"] > 0
assert stats["trainable"] < stats["total"]

classifier_params = sum(
    p.numel() for p in model.get_classifier().parameters()
)

assert stats["trainable"] == classifier_params

print("Stage 1 freezing: PASS")

Total parameters: 27,970,852
Trainable parameters: 150,724
Frozen parameters: 27,820,128
Stage 1 freezing: PASS


### Verify Stage 2 — Partial fine-tuning setup

In [21]:
stats = configure_transfer_stage(
    model,
    model_key,
    stage="partial_fine_tune"
)

print("Total parameters:", f"{stats['total']:,}")
print("Trainable parameters:", f"{stats['trainable']:,}")
print("Frozen parameters:", f"{stats['frozen']:,}")

print("\nTrainable top-level components:")

for name, module in model.named_children():
    count = sum(
        p.numel()
        for p in module.parameters()
        if p.requires_grad
    )

    if count > 0:
        print(f"{name}: {count:,}")

assert stats["trainable"] > classifier_params
assert stats["trainable"] < stats["total"]

print("\nStage 2 partial fine-tuning: PASS")

Total parameters: 27,970,852
Trainable parameters: 15,621,316
Frozen parameters: 12,349,536

Trainable top-level components:
stages: 15,470,592
head: 150,724

Stage 2 partial fine-tuning: PASS


In [23]:
import gc

del model
gc.collect()

print("ConvNeXt-Tiny released")

ConvNeXt-Tiny released


In [24]:
import torch
import timm

model_key = "xception"
dataset_key = "stanford_cars"

model_name = cfg["models"][model_key]["timm_name"]
num_classes = cfg["datasets"][dataset_key]["verified_classes"]

torch.manual_seed(cfg["random_seed"])

model = timm.create_model(
    model_name,
    pretrained=True,
    num_classes=num_classes
)

model.eval()

print("Model:", model_name)
print("Classifier:", model.get_classifier())
print("Number of classes:", num_classes)

Model: legacy_xception.tf_in1k
Classifier: Linear(in_features=2048, out_features=196, bias=True)
Number of classes: 196


In [25]:
from vehicle_models import configure_transfer_stage

stats = configure_transfer_stage(
    model,
    model_key,
    stage="head_only"
)

print("STAGE 1 — HEAD ONLY")
print(f"Total parameters: {stats['total']:,}")
print(f"Trainable parameters: {stats['trainable']:,}")
print(f"Frozen parameters: {stats['frozen']:,}")

# Expected Xception classifier parameters
assert stats["trainable"] == 401604

print("Xception Stage 1: PASS")

STAGE 1 — HEAD ONLY
Total parameters: 21,208,556
Trainable parameters: 401,604
Frozen parameters: 20,806,952
Xception Stage 1: PASS


In [26]:
stats = configure_transfer_stage(
    model,
    model_key,
    stage="partial_fine_tune"
)

print("STAGE 2 — PARTIAL FINE-TUNING")
print(f"Total parameters: {stats['total']:,}")
print(f"Trainable parameters: {stats['trainable']:,}")
print(f"Frozen parameters: {stats['frozen']:,}")

print("\nTrainable components:")

for name, module in model.named_children():
    count = sum(
        p.numel()
        for p in module.parameters()
        if p.requires_grad
    )

    if count > 0:
        print(f"{name}: {count:,}")

assert stats["trainable"] > 401604
assert stats["trainable"] < stats["total"]

print("\nXception Stage 2: PASS")

STAGE 2 — PARTIAL FINE-TUNING
Total parameters: 21,208,556
Trainable parameters: 7,189,988
Frozen parameters: 14,018,568

Trainable components:
block12: 2,039,584
conv3: 1,582,080
bn3: 3,072
conv4: 3,159,552
bn4: 4,096
fc: 401,604

Xception Stage 2: PASS


In [27]:
import gc
import torch
import timm

from vehicle_models import configure_transfer_stage

for model_key in ("xception", "convnext_tiny"):

    gc.collect()

    model_name = cfg["models"][model_key]["timm_name"]

    # Pretrained weights unnecessary for this structural check
    model = timm.create_model(
        model_name,
        pretrained=False,
        num_classes=48
    )

    print(f"\nMODEL: {model_key}")

    # Stage 1
    stage1 = configure_transfer_stage(
        model, model_key, "head_only"
    )

    classifier_params = sum(
        p.numel()
        for p in model.get_classifier().parameters()
    )

    assert stage1["trainable"] == classifier_params

    print("Stage 1 trainable:", stage1["trainable"])
    print("Stage 1: PASS")

    # Stage 2
    stage2 = configure_transfer_stage(
        model, model_key, "partial_fine_tune"
    )

    assert stage2["trainable"] > stage1["trainable"]
    assert stage2["frozen"] > 0

    print("Stage 2 trainable:", stage2["trainable"])
    print("Stage 2: PASS")

    del model

gc.collect()


MODEL: xception
Stage 1 trainable: 98352
Stage 1: PASS
Stage 2 trainable: 6886736
Stage 2: PASS

MODEL: convnext_tiny
Stage 1 trainable: 36912
Stage 1: PASS
Stage 2 trainable: 15507504
Stage 2: PASS


0